# Modelo 10 — Árvore de decisão por padrão de exames

Árvore de decisão com ganho de informação (entropia), com um submodelo para cada padrão de exames disponíveis. É o componente árvore do voto do Modelo 8, separado para ser submetido sozinho.

**Aulas usadas:** 2 (qualidade e valores ausentes), 5 (árvore de decisão e ganho de informação), 7 (validação e métricas)

**Regras da competição respeitadas:**
- O conjunto de teste **não** é usado em nenhuma escolha. Toda avaliação usa validação cruzada só no treino.
- A métrica principal é o **F1-macro**, a mesma do placar do Kaggle. Também mostramos acurácia, precisão e recall de cada classe, como pede o enunciado.

## 1. Bibliotecas e funções do projeto

In [1]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from limpeza import carregar          # limpeza e correção de unidades (Aula 2)
from padroes import *                 # PorPadrao, validação e métricas (Aulas 2 e 7)
pd.set_option('display.width', 120)

## 2. Leitura e limpeza dos dados (Aula 2 — qualidade dos dados)

A função `carregar` (arquivo `limpeza.py`) aplica regras linha a linha. Nenhuma estatística é calculada aqui; médias e medianas só são calculadas dentro do treino de cada parte da validação.

| Problema encontrado | Correção |
|---|---|
| 20 registros duplicados do mesmo paciente | removidos |
| Glicose: 44 valores entre 3,1 e 21,3 (mmol/L), o resto entre 47 e 500 (mg/dL) | valores < 35 multiplicados por 18,016 |
| HbA1c: 43 valores entre 27 e 116 (mmol/mol, padrão IFCC), o resto entre 4,7 e 13,1 (%) | valores > 20 convertidos por 0,09148 × v + 2,152 |
| Peso: 23 valores acima de 140 (libras) | divididos por 2,2046 |
| Triglicerídeos com valores extremos (até 5.310) | limitados a 1000 |
| Altura em metros em alguns registros | convertida para cm; IMC calculado |
| Pressão no formato texto `120/80` ou `?` | separada em sistólica e diastólica |
| Grafias diferentes (`F`, `female`, `FEMALE`, `Não`, `0`…) | unificadas |
| Idade 0, −1 ou 999 | recalculada por ano do exame − ano de nascimento |

In [2]:
tr, te = carregar('../dados/kaggle-training/train.csv', '../dados/kaggle-test/test.csv')
y = tr['class'].values
print('treino:', tr.shape, '| teste:', te.shape)
print('classes no treino:', dict(zip(*np.unique(y, return_counts=True))))
tr.head()

### Efeito da correção de unidades

Depois da conversão, os dois exames ficam na mesma escala e separam bem as classes. Nenhum controle tem HbA1c ≥ 6,5% (limite clínico de diabetes).

In [3]:
tr.groupby('class')[['GLICOSE', 'HBA1C']].describe().T.round(1)

class              0      1
GLICOSE count  220.0  241.0
        mean    96.9  152.9
        std     13.9   67.8
        min     68.0   47.0
        25%     86.5  104.5
        50%     95.7  135.1
        75%    105.0  183.8
        max    140.0  500.0
HBA1C   count  192.0  206.0
        mean     5.5    8.1
        std      0.5    1.9
        min      4.6    4.9
        25%      5.1    6.5
        50%      5.4    7.6
        75%      5.8    9.1
        max      6.4   13.1

## 3. Padrões de exames ausentes (Aula 2 — valores ausentes)

Quase 30% dos pacientes não têm glicose, HbA1c ou ambos. Preencher com a mediana faz o paciente parecer saudável, e era exatamente aí que os primeiros modelos erravam.

A solução é **ignorar o atributo ausente** (uma das opções da aula 2): cada paciente vai para um submodelo treinado só com os atributos que ele tem.

| Padrão | Atributos do submodelo |
|---|---|
| glicose e HbA1c | glicose, HbA1c |
| só glicose | glicose + clínicos |
| só HbA1c | HbA1c + clínicos |
| nenhum exame | só clínicos |

Clínicos: triglicerídeos, pressão sistólica, IMC, idade, histórico familiar e atividade física.

In [4]:
cont = pd.DataFrame({
    'treino': pd.Series([NOME_PADRAO[p] for p in padrao_de(tr)]).value_counts(),
    'teste': pd.Series([NOME_PADRAO[p] for p in padrao_de(te)]).value_counts()})
cont['% treino'] = (100 * cont['treino'] / len(tr)).round(1)
cont['% teste'] = (100 * cont['teste'] / len(te)).round(1)
cont

,treino,teste,% treino,% teste
glicose e HbA1c,357,222,68.7,71.2
só glicose,104,53,20.0,17.0
só HbA1c,41,26,7.9,8.3
nenhum exame,18,11,3.5,3.5


## 4. O modelo (Aula 5 — árvore de decisão)

A árvore escolhe, em cada nó, o atributo e o ponto de corte com **maior ganho de informação**: a maior redução da **entropia** das classes (Aula 5). Os atributos são numéricos, então cada teste é binário, por exemplo `HBA1C <= 6,05`.

A árvore não precisa de normalização: os cortes são comparações dentro de cada atributo.

Dois controles evitam que a árvore decore o treino:
- **profundidade máxima**: quantas perguntas em sequência ela pode fazer;
- **mínimo de pacientes por folha**: uma folha com 1 paciente é quase sempre ruído.

Testamos também o índice de Gini no lugar da entropia, e deu empate (0,897 contra 0,894). Mantivemos a entropia, que é o critério apresentado na aula.

In [5]:
from sklearn.tree import DecisionTreeClassifier, export_text

def criar_arvore(prof, folha):
    return PorPadrao(DecisionTreeClassifier(criterion='entropy', max_depth=prof,
                                            min_samples_leaf=folha, random_state=0))

## 5. Escolha do hiperparâmetro (Aula 7 — validação cruzada)

Testamos combinações de **profundidade máxima** e **mínimo de pacientes por folha**.

Protocolo: validação cruzada estratificada em **5 partes, repetida 4 vezes** (semente 2026), só no treino. Escolhemos o valor com maior F1-macro médio. A avaliação final (seção 6) usa **outra semente (777)**, para não medir o desempenho nas mesmas partes usadas na escolha.

In [6]:
linhas = []
for prof in [2, 3, 4, 5, 6, None]:
    for folha in [1, 5, 10]:
        tab, _ = validar(lambda: criar_arvore(prof, folha), tr, y, semente=2026)
        linhas.append({'profundidade': 'sem limite' if prof is None else prof, 'folha_min': folha,
                       **tab.mean().round(4).to_dict(), 'dp_f1': round(tab['f1_macro'].std(), 4)})
busca = pd.DataFrame(linhas)
busca.pivot(index='profundidade', columns='folha_min', values='f1_macro').round(3)

folha_min,1,5,10
profundidade,,,
2,0.884,0.884,0.884
3,0.890,0.891,0.891
4,0.894,0.898,0.889
5,0.886,0.895,0.891
6,0.881,0.895,0.895
sem limite,0.874,0.886,0.898


### Decisão

As melhores combinações ficam todas perto de 0,898, com diferenças na terceira casa decimal, muito menores que o desvio padrão entre as partes (cerca de 0,027). Na prática é empate. Nesse caso escolhemos a **árvore mais compacta** entre as de melhor resultado: **profundidade 4 e mínimo de 5 por folha**. Uma árvore menor é mais fácil de ler e de explicar, e tende a generalizar melhor (a aula 5 apresenta a busca por árvores compactas). É também a árvore usada no voto do Modelo 8.

In [7]:
melhor = busca.loc[busca['f1_macro'].idxmax()]
print('maior F1 da busca:', melhor['profundidade'], '| folha', melhor['folha_min'], '| F1', melhor['f1_macro'])
PROF, FOLHA = 4, 5
print('usada:', PROF, '| folha', FOLHA, '| F1',
      busca.query('profundidade == 4 and folha_min == 5')['f1_macro'].iloc[0])
criar_final = lambda: criar_arvore(PROF, FOLHA)

maior F1 da busca: sem limite | folha 10 | F1 0.8979
usada: 4 | folha 5 | F1 0.8978


### As regras aprendidas (boas para os slides)

A grande vantagem da árvore é ser legível. Abaixo, as árvores de dois submodelos treinados com todo o treino. `class: 1` é diabetes.

In [8]:
m = criar_final().fit(tr, y)
for padrao in [(1, 1), (1, 0)]:
    cols, pipe = m.modelos_[padrao]
    print(f'=== Submodelo: {NOME_PADRAO[padrao]} ===')
    print(export_text(pipe[-1], feature_names=cols, decimals=2))

=== Submodelo: glicose e HbA1c ===
|--- HBA1C <= 6.38
|   |--- GLICOSE <= 144.00
|   |   |--- GLICOSE <= 122.25
|   |   |   |--- HBA1C <= 5.42
|   |   |   |   |--- class: 0
|   |   |   |--- HBA1C >  5.42
|   |   |   |   |--- class: 0
|   |   |--- GLICOSE >  122.25
|   |   |   |--- HBA1C <= 5.95
|   |   |   |   |--- class: 1
|   |   |   |--- HBA1C >  5.95
|   |   |   |   |--- class: 0
|   |--- GLICOSE >  144.00
|   |   |--- class: 1
|--- HBA1C >  6.38
|   |--- HBA1C <= 6.45
|   |   |--- GLICOSE <= 106.00
|   |   |   |--- class: 1
|   |   |--- GLICOSE >  106.00
|   |   |   |--- class: 1
|   |--- HBA1C >  6.45
|   |   |--- class: 1

=== Submodelo: só glicose ===
|--- GLICOSE <= 120.50
|   |--- TRIGLIC <= 167.50
|   |   |--- IDADE <= 61.50
|   |   |   |--- TRIGLIC <= 83.50
|   |   |   |   |--- class: 0
|   |   |   |--- TRIGLIC >  83.50
|   |   |   |   |--- class: 0
|   |   |--- IDADE >  61.50
|   |   |   |--- IMC <= 31.05
|   |   |   |   |--- class: 0
|   |   |   |--- IMC >  31.05
|   |   

## 6. Avaliação final — validação cruzada com outra semente (Aula 7)

5 partes × 4 repetições, semente 777. Métricas da aula 7: acurácia, precisão, recall (cobertura) de cada classe e F1. O F1-macro é a média do F1 das duas classes.

In [9]:
tab, prev = validar(criar_final, tr, y, semente=777)
resumo(tab)

,média,desvio padrão
f1_macro,0.894,0.027
acuracia,0.894,0.027
precisao_macro,0.895,0.026
recall_diabetes,0.881,0.043
recall_controle,0.907,0.042


### Matriz de confusão (média por repetição; cada repetição prevê os 520 pacientes uma vez)

In [10]:
matriz_confusao(y, prev)

,previsto controle (0),previsto diabetes (1)
real controle (0),226.8,23.2
real diabetes (1),32.0,238.0


### Onde estão os erros

In [11]:
erros_por_padrao(tr, y, prev)

pacientes  erros_medios  taxa_erro
padrão          classe                                      
glicose e HbA1c controle        173          6.00       0.03
                diabetes        184         14.50       0.08
nenhum exame    controle         11          5.75       0.52
                diabetes          7          2.00       0.29
só HbA1c        controle         19          2.00       0.11
                diabetes         22          3.25       0.15
só glicose      controle         47          9.50       0.20
                diabetes         57         12.25       0.21

## 7. Treino com todos os dados e previsão do teste

O modelo final é treinado com os 520 pacientes do treino e prevê os 312 do teste. O arquivo segue o formato `id,class` do `submission_template.csv`.

In [12]:
modelo = criar_final().fit(tr, y)
prev_teste = modelo.predict(te)
sub = pd.DataFrame({'id': te['id'], 'class': prev_teste})
sub.to_csv('submissao_arvore_padroes.csv', index=False)
print('submissao_arvore_padroes.csv:', len(sub), 'linhas | diabetes previsto em', f'{prev_teste.mean():.1%}', 'do teste')
print('por padrão de exames:')
print(pd.Series(prev_teste).groupby([NOME_PADRAO[p] for p in padrao_de(te)]).agg(['size', 'mean']).round(2))

submissao_arvore_padroes.csv: 312 linhas | diabetes previsto em 47.8% do teste
por padrão de exames:
                 size  mean
glicose e HbA1c   222  0.45
nenhum exame       11  0.55
só HbA1c           26  0.54
só glicose         53  0.55


### Conferência do arquivo

In [13]:
chk = pd.read_csv('submissao_arvore_padroes.csv')
tpl = pd.read_csv('../dados/kaggle-test/test.csv', usecols=['id'])
assert list(chk.columns) == ['id', 'class']
assert chk['id'].tolist() == tpl['id'].tolist()
assert set(chk['class']) <= {0, 1}
print('OK: mesmas', len(chk), 'linhas e ordem de ids do teste; classes 0/1')

OK: mesmas 312 linhas e ordem de ids do teste; classes 0/1
